In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.audit.validation_summary (
    table_name STRING,
    bronze_count BIGINT,
    source_count BIGINT,
    validation_time TIMESTAMP,
    brnz_status STRING,
    silver_count BIGINT
)
USING DELTA;

DELETE FROM workspace.audit.validation_summary;

In [0]:
from collections import defaultdict
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

# ==========================================
# READ ENABLED CONFIGS
# ==========================================

config_df = spark.sql("""

SELECT DISTINCT *
FROM workspace.configs.migration_config
WHERE enabled = true

""")

configs = config_df.collect()

# ==========================================
# GROUP BY SOURCE + TARGET TABLE
# ==========================================

table_configs = defaultdict(list)

for cfg in configs:

    key = (
        cfg["source_table"],
        cfg["target_table"]
    )

    table_configs[key].append(cfg)

# ==========================================
# VALIDATION
# ==========================================

summary_rows = []

validation_failed = False

for (source_table, target_table), cfg_list in table_configs.items():

    print("\n===================================")
    print(f"VALIDATING : {target_table}")
    print("===================================")

    total_source_count = 0

    # ==========================================
    # GET SOURCE COUNTS
    # ==========================================

    for cfg in cfg_list:

        jdbc_url = cfg["jdbc_url"]

        properties = {
            "user": cfg["user"],
            "password": cfg["password"],
            "driver": cfg["driver"]
        }

        query = f"(SELECT COUNT(*) AS cnt FROM {source_table}) src"

        try:

            source_count = (
                spark.read.jdbc(
                    url=jdbc_url,
                    table=query,
                    properties=properties
                )
                .first()["cnt"]
            )

            print(f"SUCCESS : {jdbc_url}")
            print(f"COUNT   : {source_count}")

            total_source_count += source_count

        except Exception as e:

            print(f"FAILED : {jdbc_url}")
            print(str(e))

            validation_failed = True

    # ==========================================
    # GET BRONZE COUNT
    # ==========================================

    try:

        bronze_count = (
            spark.sql(f"""
                SELECT COUNT(*) AS cnt
                FROM {target_table}
            """)
            .first()["cnt"]
        )

    except Exception as e:

        print(f"FAILED TO READ TARGET TABLE : {target_table}")
        print(str(e))

        bronze_count = 0

        validation_failed = True

    # ==========================================
    # VALIDATION STATUS
    # ==========================================

    status = (
        "PASS"
        if total_source_count == bronze_count
        else "FAIL"
    )

    if status == "FAIL":
        validation_failed = True

    # ==========================================
    # PRINT SUMMARY
    # ==========================================

    print("-----------------------------------")
    print(f"TOTAL SOURCE COUNT : {total_source_count}")
    print(f"BRONZE COUNT       : {bronze_count}")
    print(f"STATUS             : {status}")
    print("-----------------------------------")

    # ==========================================
    # BUILD SUMMARY ROW
    # ==========================================

    summary_rows.append(
        Row(
            table_name=target_table,
            bronze_count=bronze_count,
            source_count=total_source_count,
            brnz_status=status,
            silver_count=0
           
        )
    )
# ==========================================
# CREATE SUMMARY DATAFRAME
# ==========================================

summary_df = spark.createDataFrame(summary_rows)

summary_df = summary_df.withColumn(
    "validation_time",
    current_timestamp()
)

# ==========================================
# WRITE AUDIT TABLE
# ==========================================

(
    summary_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.audit.validation_summary")
)

print("\nValidation summary inserted")

# ==========================================
# FAIL WORKFLOW IF ANY VALIDATION FAILED
# ==========================================

if validation_failed:

    raise Exception("""
    Validation failed for one or more tables.
    Check workspace.audit.validation_summary
    """)

print("\nALL VALIDATIONS PASSED")

In [0]:
%sql
select * from workspace.audit.validation_summary

In [0]:
# from collections import defaultdict

# # Read enabled configs
# config_df = spark.sql("""

# SELECT *
# FROM workspace.configs.migration_config
# WHERE enabled = true

# """)

# configs = config_df.collect()

# # Group configs by target_table
# table_configs = defaultdict(list)

# for cfg in configs:
#     table_configs[cfg["target_table"]].append(cfg)

# validation_errors = []

# # Loop through each target table
# for target_table, cfg_list in table_configs.items():

#     print(f"\n========== Validating {target_table} ==========")

#     total_source_count = 0

#     # Get source counts from all sources for this table
#     for cfg in cfg_list:

#         source_name = cfg["source_name"]
#         jdbc_url = cfg["jdbc_url"]
#         source_table = cfg["source_table"]

#         properties = {
#             "user": cfg["user"],
#             "password": cfg["password"],
#             "driver": cfg["driver"]
#         }

#         query = f"(select count(*) as cnt from {source_table}) src"

#         source_count = (
#             spark.read.jdbc(
#                 url=jdbc_url,
#                 table=query,
#                 properties=properties
#             )
#             .collect()[0]["cnt"]
#         )

#         print(f"{source_name} ({source_table}) => {source_count}")

#         total_source_count += source_count

#     # Bronze table count
#     bronze_count = (
#         spark.sql(f"""
#             SELECT COUNT(*) as cnt
#             FROM {target_table}
#         """)
#         .collect()[0]["cnt"]
#     )

#     print(f"Total Source Count : {total_source_count}")
#     print(f"Bronze Count       : {bronze_count}")

#     # Compare counts
#     if total_source_count != bronze_count:

#         validation_errors.append(
#             f"""
#             Table: {target_table}
#             Source Count = {total_source_count}
#             Bronze Count = {bronze_count}
#             """
#         )

# # Final validation result
# if validation_errors:

#     error_message = "\n".join(validation_errors)

#     raise Exception(f"""
#     VALIDATION FAILED

#     {error_message}
#     """)

# print("\nALL TABLE VALIDATIONS PASSED")